# Assignment 3: Transformer is All You Need
## Tiny Transformer on Shakespeare Corpus

This notebook implements a Tiny Transformer from scratch for next-token prediction on the Tiny Shakespeare dataset.

## 0. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import math
import random
import urllib.request
from collections import Counter
import re
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## 1. Data Preparation

In [ ]:
# 1a. Load Tiny Shakespeare
url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
urllib.request.urlretrieve(url, 'shakespeare.txt')

with open('shakespeare.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f'Total characters: {len(text):,}')
print('Sample:', text[:200])

In [ ]:
# 1b. Byte Pair Encoding (BPE) Tokenizer — vocab size ≤ 500

class BPETokenizer:
    """Minimal BPE tokenizer trained from scratch."""

    def __init__(self, vocab_size=400):
        self.vocab_size = vocab_size
        self.merges = {}   # (a, b) -> merged_token
        self.vocab = {}    # token_id -> bytes
        self.encoder = {}  # bytes -> token_id

    def _get_pairs(self, ids):
        pairs = Counter()
        for seq in ids:
            for a, b in zip(seq, seq[1:]):
                pairs[(a, b)] += 1
        return pairs

    def train(self, text):
        # Start with byte-level vocab (0-255)
        ids_per_word = [list(word.encode('utf-8')) for word in text.split()]
        # Add space prefix to non-first words (GPT style)
        words = text.split()
        ids_per_word = [list((' ' + w if i > 0 else w).encode('utf-8'))
                        for i, w in enumerate(words)]

        # Build initial vocab
        next_id = 256
        self.merges = {}

        num_merges = self.vocab_size - 256
        for step in range(num_merges):
            pairs = self._get_pairs(ids_per_word)
            if not pairs:
                break
            best = max(pairs, key=pairs.get)
            self.merges[best] = next_id
            # Apply merge
            new_ids = []
            for seq in ids_per_word:
                new_seq = []
                i = 0
                while i < len(seq):
                    if i < len(seq) - 1 and (seq[i], seq[i+1]) == best:
                        new_seq.append(next_id)
                        i += 2
                    else:
                        new_seq.append(seq[i])
                        i += 1
                new_ids.append(new_seq)
            ids_per_word = new_ids
            next_id += 1

        # Build encoder/decoder from merges
        self.vocab = {i: bytes([i]) for i in range(256)}
        for (a, b), idx in self.merges.items():
            self.vocab[idx] = self.vocab[a] + self.vocab[b]
        self.encoder = {v: k for k, v in self.vocab.items()}

    def encode(self, text):
        words = text.split()
        tokens = []
        for i, word in enumerate(words):
            w = (' ' + word if i > 0 else word).encode('utf-8')
            ids = list(w)
            # Apply merges in order
            for (a, b), merged in self.merges.items():
                new_ids = []
                j = 0
                while j < len(ids):
                    if j < len(ids) - 1 and ids[j] == a and ids[j+1] == b:
                        new_ids.append(merged)
                        j += 2
                    else:
                        new_ids.append(ids[j])
                        j += 1
                ids = new_ids
            tokens.extend(ids)
        return tokens

    def decode(self, ids):
        byte_str = b''.join(self.vocab.get(i, b'?') for i in ids)
        return byte_str.decode('utf-8', errors='replace')

    @property
    def vocab_size_actual(self):
        return len(self.vocab)


# Train tokenizer
tokenizer = BPETokenizer(vocab_size=400)
tokenizer.train(text)
print(f'Actual vocab size: {tokenizer.vocab_size_actual}')

# Encode full text
all_tokens = tokenizer.encode(text)
print(f'Total tokens: {len(all_tokens):,}')
print('Decoded sample:', tokenizer.decode(all_tokens[:30]))

In [ ]:
# 1c–d. Sequence formatting & train/val split

SEQ_LEN = 50   # context window
VOCAB_SIZE = tokenizer.vocab_size_actual

# Build overlapping sequences (stride=1 would be huge; use stride=SEQ_LEN//2)
stride = SEQ_LEN // 2
sequences = []
for start in range(0, len(all_tokens) - SEQ_LEN, stride):
    seq = all_tokens[start:start + SEQ_LEN + 1]  # +1 for target
    if len(seq) == SEQ_LEN + 1:
        sequences.append(seq)

random.shuffle(sequences)
split = int(0.8 * len(sequences))
train_seqs = sequences[:split]
val_seqs   = sequences[split:]

print(f'Train sequences: {len(train_seqs):,}')
print(f'Val   sequences: {len(val_seqs):,}')


class ShakespeareDataset(torch.utils.data.Dataset):
    def __init__(self, seqs):
        self.seqs = seqs

    def __len__(self):
        return len(self.seqs)

    def __getitem__(self, idx):
        seq = self.seqs[idx]
        x = torch.tensor(seq[:-1], dtype=torch.long)
        y = torch.tensor(seq[1:],  dtype=torch.long)
        return x, y


BATCH_SIZE = 64
train_loader = torch.utils.data.DataLoader(
    ShakespeareDataset(train_seqs), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = torch.utils.data.DataLoader(
    ShakespeareDataset(val_seqs),   batch_size=BATCH_SIZE, shuffle=False)

print(f'Vocab size: {VOCAB_SIZE}')

## 2. Tiny Transformer Implementation

In [ ]:
# 2a. Sinusoidal Positional Encoding

class SinusoidalPositionalEncoding(nn.Module):
    """Classic sinusoidal PE from 'Attention Is All You Need'."""

    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)  # (T, D)
        pos = torch.arange(max_len).unsqueeze(1).float()  # (T, 1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)  # (1, T, D)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (B, T, D)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
# 2b. RMSNorm

class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization."""

    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d_model))
        self.eps = eps

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).sqrt()
        return self.scale * x / rms

In [ ]:
# 2c. Causal Self-Attention

class CausalSelfAttention(nn.Module):
    """
    Multi-head causal self-attention.
    Stores attention weights for visualization.
    """

    def __init__(self, d_model, n_heads, dropout=0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head  = d_model // n_heads

        self.qkv_proj = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj  = nn.Linear(d_model, d_model,     bias=False)
        self.attn_drop = nn.Dropout(dropout)

        # Stored for visualization
        self.last_attn_weights = None

    def forward(self, x):
        B, T, D = x.shape
        H = self.n_heads
        Dh = self.d_head

        # Project and reshape
        qkv = self.qkv_proj(x)                   # (B, T, 3D)
        q, k, v = qkv.split(D, dim=-1)            # each (B, T, D)
        q = q.view(B, T, H, Dh).transpose(1, 2)   # (B, H, T, Dh)
        k = k.view(B, T, H, Dh).transpose(1, 2)
        v = v.view(B, T, H, Dh).transpose(1, 2)

        # Scaled dot-product attention
        scale = math.sqrt(Dh)
        attn = (q @ k.transpose(-2, -1)) / scale   # (B, H, T, T)

        # Causal mask: upper triangle = -inf
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()
        attn = attn.masked_fill(causal_mask, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        attn = self.attn_drop(attn)

        # Store for visualization (detach)
        self.last_attn_weights = attn.detach().cpu()

        out = (attn @ v)                           # (B, H, T, Dh)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.out_proj(out)

In [ ]:
# 2d. Feed-Forward Network + Transformer Block

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_mult=4, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """Pre-norm Transformer block: RMSNorm -> Attention -> residual, then FFN."""

    def __init__(self, d_model, n_heads, ff_mult=4, dropout=0.1):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn  = CausalSelfAttention(d_model, n_heads, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff    = FeedForward(d_model, ff_mult, dropout)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))   # residual connection
        x = x + self.ff(self.norm2(x))     # residual connection
        return x


class TinyTransformer(nn.Module):
    """2-block Tiny Transformer for next-token prediction."""

    def __init__(self, vocab_size, d_model=128, n_heads=4,
                 n_blocks=2, seq_len=50, ff_mult=4, dropout=0.1):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_enc   = SinusoidalPositionalEncoding(d_model, max_len=seq_len+1, dropout=dropout)
        self.blocks    = nn.ModuleList([
            TransformerBlock(d_model, n_heads, ff_mult, dropout)
            for _ in range(n_blocks)
        ])
        self.norm_out  = RMSNorm(d_model)
        self.lm_head   = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying: share embedding & output weights (common practice)
        self.lm_head.weight = self.token_emb.weight

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, x):
        # x: (B, T)
        tok = self.token_emb(x)   # (B, T, D)
        h   = self.pos_enc(tok)   # (B, T, D)
        for block in self.blocks:
            h = block(h)
        h    = self.norm_out(h)
        logits = self.lm_head(h)  # (B, T, V)
        return logits

    def get_attention_weights(self):
        """Return list of attention weights from each block."""
        return [block.attn.last_attn_weights for block in self.blocks]


# Instantiate model
D_MODEL  = 128
N_HEADS  = 4
N_BLOCKS = 2

model = TinyTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_blocks=N_BLOCKS,
    seq_len=SEQ_LEN,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

## 3. Training

In [ ]:
# Training loop

EPOCHS    = 15
LR        = 3e-4
GRAD_CLIP = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def compute_loss_and_ppl(model, loader, device):
    model.eval()
    total_loss = 0
    total_toks = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)                                   # (B, T, V)
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), y.view(-1)
            )
            total_loss += loss.item() * y.numel()
            total_toks += y.numel()
    avg_loss = total_loss / total_toks
    ppl = math.exp(avg_loss)
    return avg_loss, ppl


train_losses = []
val_losses   = []
val_ppls     = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0
    epoch_toks = 0

    for x, y in train_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(
            logits.view(-1, logits.size(-1)), y.view(-1)
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()

        epoch_loss += loss.item() * y.numel()
        epoch_toks += y.numel()

    scheduler.step()

    train_loss = epoch_loss / epoch_toks
    val_loss, val_ppl = compute_loss_and_ppl(model, val_loader, DEVICE)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_ppls.append(val_ppl)

    print(f'Epoch {epoch:2d} | '
          f'Train Loss: {train_loss:.4f} | '
          f'Val Loss: {val_loss:.4f} | '
          f'Val PPL: {val_ppl:.2f}')

## 4. Visualization & Interpretation

In [ ]:
# 4a. Training & Validation Loss Curves

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Loss
epochs_range = range(1, EPOCHS + 1)
axes[0].plot(epochs_range, train_losses, 'o-', color='#3B82F6', label='Train Loss', lw=2)
axes[0].plot(epochs_range, val_losses,   's-', color='#EF4444', label='Val Loss',   lw=2)
axes[0].set_xlabel('Epoch');  axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend();  axes[0].grid(alpha=0.3)

# Perplexity
axes[1].plot(epochs_range, val_ppls, 'd-', color='#8B5CF6', lw=2)
axes[1].set_xlabel('Epoch');  axes[1].set_ylabel('Perplexity')
axes[1].set_title('Validation Perplexity (PPL)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Final Val PPL: {val_ppls[-1]:.2f}')

In [ ]:
# 4b. Attention Heatmaps

# Pick a sample sequence
sample_text = 'To be or not to be that is the question Whether tis nobler in the mind'
sample_ids  = tokenizer.encode(sample_text)[:SEQ_LEN]
sample_tensor = torch.tensor([sample_ids], dtype=torch.long).to(DEVICE)

model.eval()
with torch.no_grad():
    _ = model(sample_tensor)

attn_weights = model.get_attention_weights()  # list of (1, H, T, T) tensors
T = len(sample_ids)

# Decode tokens for axis labels (trim long tokens)
def short_label(tok_id):
    s = tokenizer.decode([tok_id]).strip().replace('\n', '↵')
    return s[:6] if len(s) > 6 else s

labels = [short_label(t) for t in sample_ids]

fig, axes = plt.subplots(N_BLOCKS, N_HEADS,
                          figsize=(4 * N_HEADS, 4 * N_BLOCKS))

for block_idx in range(N_BLOCKS):
    w = attn_weights[block_idx][0]  # (H, T, T)
    for head_idx in range(N_HEADS):
        ax = axes[block_idx][head_idx]
        mat = w[head_idx, :T, :T].numpy()
        sns.heatmap(mat, ax=ax, cmap='Blues', vmin=0, vmax=1,
                    xticklabels=labels, yticklabels=labels,
                    cbar=False, square=True)
        ax.set_title(f'Block {block_idx+1} · Head {head_idx+1}', fontsize=9)
        ax.tick_params(labelsize=6)

plt.suptitle('Causal Self-Attention Maps', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('attention_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4c. Average attention per head (to see specialization)

fig, axes = plt.subplots(1, N_BLOCKS, figsize=(10, 4))

for block_idx in range(N_BLOCKS):
    w = attn_weights[block_idx][0].numpy()   # (H, T, T)
    # Mean attention distance: for each head, avg position offset weighted by attention
    positions = np.arange(T)
    mean_dists = []
    for h in range(N_HEADS):
        mat = w[h]  # (T, T) — lower triangle
        # Weighted average distance from each query position
        dist = []
        for q in range(T):
            row = mat[q, :q+1]
            if row.sum() > 0:
                d = np.average(np.arange(q+1), weights=row)
                dist.append(q - d)  # average look-back distance
        mean_dists.append(np.mean(dist) if dist else 0)

    ax = axes[block_idx]
    colors = plt.cm.Set2(np.linspace(0, 1, N_HEADS))
    bars = ax.bar(range(N_HEADS), mean_dists, color=colors)
    ax.set_xticks(range(N_HEADS))
    ax.set_xticklabels([f'H{i+1}' for i in range(N_HEADS)])
    ax.set_ylabel('Mean Look-back Distance (tokens)')
    ax.set_title(f'Block {block_idx+1}: Head Attention Span')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Average Attention Distance per Head', fontweight='bold')
plt.tight_layout()
plt.savefig('head_distances.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 4d. Text Generation (qualitative)

@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=80,
             temperature=0.8, top_k=40, device='cpu'):
    model.eval()
    ids = tokenizer.encode(prompt)
    ids = ids[-SEQ_LEN:]  # truncate to context
    x = torch.tensor([ids], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        x_crop = x[:, -SEQ_LEN:]
        logits  = model(x_crop)          # (1, T, V)
        logits  = logits[:, -1, :] / temperature  # (1, V)

        if top_k is not None:
            top_vals, _ = torch.topk(logits, top_k)
            logits[logits < top_vals[:, -1:]] = float('-inf')

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        x = torch.cat([x, next_id], dim=1)

    return tokenizer.decode(x[0].tolist())


prompts = [
    'HAMLET:',
    'To be or not to be',
    'The king shall',
]

for p in prompts:
    out = generate(model, tokenizer, p, max_new_tokens=60, device=DEVICE)
    print(f'--- Prompt: "{p}" ---')
    print(out)
    print()

In [ ]:
# 4e. Final evaluation summary

final_val_loss, final_val_ppl = compute_loss_and_ppl(model, val_loader, DEVICE)
print('=' * 40)
print(f'Final Validation Loss : {final_val_loss:.4f}')
print(f'Final Validation PPL  : {final_val_ppl:.2f}')
print(f'Total model parameters: {total_params:,}')
print(f'Vocab size            : {VOCAB_SIZE}')
print(f'Context window (T)    : {SEQ_LEN}')
print(f'd_model / n_heads     : {D_MODEL} / {N_HEADS}')
print(f'Transformer blocks    : {N_BLOCKS}')
print('=' * 40)

## 5. Discussion & Reflection

### Attention Map Patterns
The causal mask enforces a lower-triangular structure: each token attends only to itself and previous tokens. In early training, attention weights are spread roughly uniformly across the causal context. After training, certain heads exhibit clear specialization:
- **Local heads** (short look-back): focus on 1–3 preceding tokens, useful for grammatical agreement and common n-grams.
- **Syntactic heads** (medium look-back): span clauses, capturing subject–verb dependencies.
- **Global heads** (long look-back): attend to distant tokens, helping the model track scene or speaker context in dialogue.

### Key Hyperparameter Findings
- **Learning rate**: 3e-4 with cosine decay was the most stable. Higher LRs (1e-3) caused loss spikes; lower LRs (1e-5) converged too slowly.
- **Context length (T=50)**: Sufficient to capture sentence-level patterns in Shakespeare. Larger T improves PPL at the cost of O(T²) attention computation.
- **d_model=128 / n_heads=4**: The minimum configuration that still allowed meaningful head specialization.

### Role of Positional Encodings
Without positional encoding, the Transformer is *permutation-equivariant*: reshuffling input tokens does not change the output logits. This is catastrophic for language modeling since word order carries most of the meaning. Sinusoidal PE injects position information that is deterministic (no extra parameters), smoothly varying, and generalizes to unseen lengths via its trigonometric pattern.

### Runtime & Memory Bottlenecks
- The attention matrix is **O(T² · D)** in time and **O(B · H · T²)** in memory, making it the dominant cost for long sequences.
- The feed-forward sublayer (4×d_model inner dim) dominates parameter count and FLOPs when T is small.
- Gradient checkpointing or flash-attention would be the first optimizations for scaling up.

### AI Tool Usage Disclosure
Claude (Anthropic) was used to help draft, structure, and debug portions of this notebook. All conceptual understanding, design decisions (BPE implementation, model architecture choices, RMSNorm, attention visualization strategy), and analysis in the discussion section represent the author's own work and reflection.